# 02 — Find the Pool by Its Job

**Target:** Apache Spark 4.2.0; Python 3.10+. No external JDK tools are required for this lab.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
Which JVM threads belong to task execution, RPC, shuffle and background maintenance?

**Why this matters:** thread count alone does not identify a leak or useful parallelism. A thread dump gives names and stack traces that help map activity to a subsystem.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
import os, sys, tempfile, threading, time
os.environ['PYSPARK_PYTHON'] = sys.executable
spark = SparkSession.builder.master('local[2]').appName('thread-lab-02').getOrCreate()
thread_mx = spark.sparkContext._jvm.java.lang.management.ManagementFactory.getThreadMXBean()
print('Driver PID:', os.getpid(), 'Spark:', spark.version)
print('Spark UI:', spark.sparkContext.uiWebUrl or 'disabled')

def capture_dump(label):
    infos = thread_mx.dumpAllThreads(True, True)
    path = os.path.join(tempfile.gettempdir(), f'spark-thread-lab-{label}-{time.time_ns()}.txt')
    with open(path, 'w', encoding='utf-8') as dump_file:
        dump_file.write('\n'.join(str(info) for info in infos))
    return path

idle_dump = capture_dump('idle')
print('Idle thread dump saved to:', idle_dump)


In [ ]:
# Guarantee an observable active interval, while keeping the input tiny and bounded.
import threading as threading_lib

@F.udf(returnType=T.LongType())
def briefly_slow(value):
    time.sleep(0.05)
    return value

query_errors = []
def run_shuffle():
    try:
        (spark.range(0, 120).select(briefly_slow('id').alias('id'))
         .repartition(4, F.col('id') % 11)
         .groupBy(F.col('id') % 13).count().collect())
    except Exception as error:
        query_errors.append(error)

query_thread = threading_lib.Thread(target=run_shuffle, name='bounded-spark-shuffle')
query_thread.start()
active_dumps = []
while query_thread.is_alive() and len(active_dumps) < 3:
    active_dumps.append(capture_dump('active'))
    time.sleep(0.5)
query_thread.join()
if query_errors:
    raise query_errors[0]
print(f'Captured {len(active_dumps)} active dump(s):', active_dumps)


## Compare the dumps
This cell captures one idle JVM dump, then triggers a tiny shuffle with a short Python UDF delay. While that bounded action is active, it asks the JVM's own `ThreadMXBean` for up to three thread dumps and saves them under the operating system temporary directory. Open an `active` dump and compare names and stack traces with the `idle` dump.

**Expected:** task-running threads should not be confused with Netty event loops or RPC dispatchers. During a Python UDF, JVM task threads may spend time in PythonRunner/Python worker communication; the worker itself is a separate Python process and will not appear as a JVM thread.

**Why:** thread pools serve separate queues and responsibilities. A dump is a snapshot, so a very short operation can vanish between manual commands; the bounded delay makes this lab observable without creating a large workload. Using the JVM management API also avoids PID mismatches between a Python container and its remote Spark JVM. Spark configuration for shuffle/RPC pools is release-sensitive; inspect the Spark 4.2.0 configuration reference before tuning.

**Safety:** only inspect processes you own or are authorized to inspect. The workload is 120 rows with a 50 ms UDF delay and at most three dumps. Stop the session after capturing and comparing them.

In [ ]:
spark.stop()